# Data split

In [ ]:
import numpy as np

# group_ids: one entry per window, e.g. "a/gear/gear_1"
group_ids = np.asarray(group_ids)

participant_ids = np.array([
    str(group_id).split("/")[0]
    for group_id in group_ids
])

print("Participants:", np.unique(participant_ids))
print("Windows per participant:")
for participant in np.unique(participant_ids):
    print(participant, np.sum(participant_ids == participant))

In [ ]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, temp_idx = next(
    splitter.split(X, y, groups=participant_ids)
)

X_train, y_train = X[train_idx], y[train_idx]
X_temp, y_temp = X[temp_idx], y[temp_idx]

# Split the remaining participants into validation and test
temp_participants = np.unique(participant_ids[temp_idx])

rng = np.random.default_rng(42)
rng.shuffle(temp_participants)

n_val = max(1, len(temp_participants) // 2)
val_participants = set(temp_participants[:n_val])
test_participants = set(temp_participants[n_val:])

val_mask = np.isin(participant_ids, list(val_participants))
test_mask = np.isin(participant_ids, list(test_participants))

X_val, y_val = X[val_mask], y[val_mask]
X_test, y_test = X[test_mask], y[test_mask]

# Model

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

n_timesteps = 128   # 2.56 s at 50 Hz
n_features = 6      # acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z
num_classes = 6     # change to your actual number of activities

normalizer = layers.Normalization(axis=-1)
normalizer.adapt(X_train)  # X_train must already be participant-split

model = models.Sequential([
    layers.Input(shape=(n_timesteps, n_features)),

    normalizer,

    layers.Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        activation="relu"
    ),

    layers.Conv1D(
        filters=64,
        kernel_size=5,
        padding="same",
        activation="relu"
    ),

    layers.MaxPooling1D(pool_size=2),

    layers.SpatialDropout1D(rate=0.2),

    layers.LSTM(units=64),

    layers.Dropout(rate=0.4),

    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# Training

In [ ]:
history = model.fit(
    X_train,
    y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=64,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=5,
            restore_best_weights=True
        )
    ]
)